<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://mng.bz/lZ5B">Sebastian Raschka《从零构建推理模型》</a>一书的配套代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第 6 章：练习解答

本 Notebook 使用的程序包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

&nbsp;
## 练习 6.1：添加格式感知的奖励塑形

- 如果找不到 `\boxed{}` 格式的答案，可以使用第 3 章编写的 `fallback="number_then_full"` 回退方式，分配部分奖励（分数 0.5），如下所示：

In [ ]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def partial_reward_rlvr(answer_text, ground_truth):
    
    # 1) Try to extract a boxed answer
    boxed = extract_final_candidate(
        answer_text, fallback=None
    )
    if boxed:
        correct = grade_answer(boxed, ground_truth)
        return 1.0 if correct else 0.0

    # 2) If no boxed answer is found, look for number
    unboxed = extract_final_candidate(
        answer_text, fallback="number_then_full"
    )
    if unboxed:
        correct = grade_answer(unboxed, ground_truth)
        return 0.5 if correct else 0.0

    return 0.0

- 在第 6 章代码中使用相同设置进行训练时，部分奖励版本的准确率为 37.8%，低于标准 GRPO 设置的 47.4%，尽管两者平均使用的 token 数量相近

| # | 方法 | 步数 | 最大 token 数 | Rollout 数 | 准确率 | 平均 token 数 |
|---|------|------|------------|------------|--------|----------------|
| 1 | GRPO（第 6 章） | 50 | 512 | 8 | 47.4% | 586.11 |
| 2 | GRPO 部分奖励（练习 6.1） | 50 | 512 | 8 | 37.8% | 550.33 |

&nbsp;
## 练习 6.2：优势值为零的情况

- 如果所有奖励都相等（例如全部为 0 或全部为 1），优势值都会是 0，因为减去平均值会消除共同的奖励值，只留下零；下面可以对此进行演示

In [3]:
import torch

rollout_rewards = [0., 0., 0., 0.]
rewards = torch.tensor(rollout_rewards)
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([0., 0., 0., 0.])


In [4]:
rollout_rewards = [1., 1., 1., 1.]
rewards = torch.tensor(rollout_rewards)
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([0., 0., 0., 0.])


- 现在，如果所有优势值都是 0，损失也会是 0，因为损失将优势值与对数概率相乘，乘以 0 会消除这部分贡献

```python
pg_loss = -(advantages.detach() * logps).mean()
```

- 因此，对于该提示词，策略梯度为 0，模型参数不会更新

- 这种行为是有意设计的；如果所有 rollout 都同样糟糕或同样优秀，就没有相对信号告诉模型应该强化或抑制哪种行为
- 直观地说，如果模型把所有问题都答对了，就没有必要更新它
- 反过来，如果模型把所有问题都答错了，我们也不希望更新模型来强化这种行为